### TaskGroup & Structured Concurrency

In [ ]:
import asyncio
import time

start_time = time.monotonic()

def log(message: str) -> None:
    elapsed = time.monotonic() - start_time
    print(f"[{elapsed:6.2f}s] {message}")

In [ ]:
async def make_item(item: str = "coffee", brew: float = 2.0, fail: bool = False) -> str:
    try:
        await asyncio.sleep(brew)
    except asyncio.CancelledError:
        log(f"{item}: thrown out")
        raise
    if fail:
        raise RuntimeError(f"out of {item}")
    return item

**Happy path**

In [ ]:
start_time = time.monotonic()

async with asyncio.TaskGroup() as group:
    task_espresso = group.create_task(make_item("espresso", brew=0.5))
    task_toast = group.create_task(make_item("toast", brew=1.0))
    task_latte = group.create_task(make_item("latte", brew=1.5))

log(f"espresso: {task_espresso.result()}")
log(f"toast:    {task_toast.result()}")
log(f"latte:    {task_latte.result()}")

In [ ]:
start_time = time.monotonic()

results = await asyncio.gather(
    make_item("espresso", brew=0.3, fail=True),
    make_item("toast", brew=1.0),
    make_item("latte", brew=1.0),
    return_exceptions=True,
)
for outcome in results:
    log(f"outcome: {outcome!r}")

**One fails — siblings get cancelled**

In [ ]:
start_time = time.monotonic()

try:
    async with asyncio.TaskGroup() as group:
        group.create_task(make_item("espresso", brew=0.3, fail=True))
        group.create_task(make_item("toast", brew=1.5))
        group.create_task(make_item("latte", brew=1.5))
except* RuntimeError as error_group:
    for error in error_group.exceptions:
        log(f"kitchen says: {error}")

**Several failures at once**

In [ ]:
start_time = time.monotonic()

try:
    async with asyncio.TaskGroup() as group:
        group.create_task(make_item("espresso", brew=0.3, fail=True))
        group.create_task(make_item("toast", brew=0.3, fail=True))
        group.create_task(make_item("latte", brew=1.5))
except* RuntimeError as kitchen_errors:
    log(f"{len(kitchen_errors.exceptions)} kitchen problems")
    for error in kitchen_errors.exceptions:
        log(f"  - {error}")

---
- `TaskGroup` waits for every child; if one raises, siblings are cancelled.
- Errors arrive as an `ExceptionGroup`; catch with `except* SomeError`.
- `gather(return_exceptions=True)` is the opposite: never cancels, just collects.